# MockMate AI - LLaMA 3.2 3B 1-Click Fine-Tuning Notebook

This notebook runs 3 full epochs of QLoRA fine-tuning on the MockMate AI dataset split (`train.jsonl` and `validation.jsonl`).
It exports a `q4_k_m` GGUF file (`mockmate-llama-q4_k_m.gguf`), computes SHA256 hashes, packages a `train_run.json` execution summary, and downloads the final zip package.

## Step 1: Check GPU Environment

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## Step 2: Install Fine-Tuning Dependencies

In [ ]:
!pip install -U unsloth "datasets>=2.18.0" "transformers>=4.41.2" "trl>=0.8.6" "peft>=0.11.1" "accelerate>=0.31.0" "bitsandbytes>=0.43.1" "huggingface_hub"

## Step 3: Hugging Face Authentication

In [ ]:
import getpass
from huggingface_hub import login
hf_token = getpass.getpass('Enter your Hugging Face Token (Read access required for LLaMA 3.2): ')
if hf_token.strip():
    login(token=hf_token.strip())
    print('Logged in to Hugging Face successfully.')
else:
    print('No token provided. Note: Gated models require authentication.')

## Step 4: Upload Training Splits
Upload `train.jsonl` and `validation.jsonl` from `mockmate/training/data/processed/splits/`.

In [ ]:
from google.colab import files
import os
print('Please upload train.jsonl and validation.jsonl:')
uploaded = files.upload()
for fn in ['train.jsonl', 'validation.jsonl']:
    if not os.path.exists(fn):
        raise FileNotFoundError(f'Missing required upload: {fn}')
print('Uploads verified:', list(uploaded.keys()))

## Step 5: Load LLaMA 3.2 3B Model & Setup QLoRA

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048
model_name = 'unsloth/Llama-3.2-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('Model and LoRA adapter initialized successfully.')

## Step 6: Train Model for 3 Epochs

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

ds = load_dataset('json', data_files={'train': 'train.jsonl', 'validation': 'validation.jsonl'})
train_count = len(ds['train'])
val_count = len(ds['validation'])

batch_size = 2
grad_accum = 4
epochs = 3
effective_batch = batch_size * grad_accum
steps_per_epoch = max(1, train_count // effective_batch)
max_steps = steps_per_epoch * epochs

print('=======================================================')
print(' PRE-TRAINING SETUP & DATASET SUMMARY')
print('=======================================================')
print(f'Train Row Count      : {train_count}')
print(f'Validation Row Count : {val_count}')
print(f'Batch Size per Device: {batch_size}')
print(f'Grad Accumulation    : {grad_accum}')
print(f'Effective Batch Size : {effective_batch}')
print(f'Steps per Epoch      : {steps_per_epoch}')
print(f'Total Max Steps (3x) : {max_steps}')
print('=======================================================\n')

sft_config = SFTConfig(
    output_dir='./mockmate-llama-adapter',
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=grad_accum,
    warmup_steps=10,
    max_steps=max_steps,
    learning_rate=2e-4,
    logging_steps=10,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
    report_to='none',
    max_length=max_seq_length,
    packing=False,
    eval_strategy='steps',
    eval_steps=max(10, max_steps // 5),
    save_strategy='steps',
    save_steps=max(10, max_steps // 5),
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    args=sft_config,
)

trainer_stats = trainer.train()
print('Training complete!')


## Step 7: Export LoRA Adapter & GGUF Q4_K_M Artifacts

In [ ]:
import os, glob, shutil, hashlib, json

adapter_dir = './mockmate-llama-adapter'
gguf_dir = './mockmate-llama-gguf'
target_gguf_filename = 'mockmate-llama-q4_k_m.gguf'
target_gguf_path = os.path.join(gguf_dir, target_gguf_filename)

os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('Saved LoRA adapter to:', adapter_dir)

os.makedirs(gguf_dir, exist_ok=True)
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method='q4_k_m')

print('\n--- GLOBBING EXPORTED GGUF FILES ---')
found_ggufs = glob.glob(os.path.join(gguf_dir, '*.gguf'))
if not found_ggufs:
    found_ggufs = glob.glob(os.path.join(gguf_dir, '**', '*.gguf'), recursive=True)

if not found_ggufs:
    raise FileNotFoundError(f'CRITICAL: No *.gguf files were produced in {gguf_dir} after save_pretrained_gguf!')

matched_q4_path = None
for fpath in found_ggufs:
    fsize = os.path.getsize(fpath)
    print(f'  Found GGUF: {fpath} | Size: {fsize / (1024*1024):.2f} MB')
    if 'q4_k_m' in fpath.lower() or not matched_q4_path:
        matched_q4_path = fpath

if matched_q4_path and matched_q4_path != target_gguf_path:
    shutil.copyfile(matched_q4_path, target_gguf_path)
    print(f'Copied/Renamed {matched_q4_path} -> {target_gguf_path}')

def calc_sha256(path):
    if not os.path.exists(path): return None
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8192): h.update(chunk)
    return h.hexdigest()

gguf_sha256 = calc_sha256(target_gguf_path)
print('Target GGUF Path  :', target_gguf_path)
print('Target GGUF SHA256:', gguf_sha256)


## Step 8: Package Manifest & Download Zip Artifact

In [ ]:
from google.colab import files

log_history = trainer.state.log_history
train_losses = [x['loss'] for x in log_history if 'loss' in x]
eval_losses = [x['eval_loss'] for x in log_history if 'eval_loss' in x]

manifest = {
    'model': model_name,
    'epochs': epochs,
    'max_steps': max_steps,
    'dataset': {
        'train_rows': train_count,
        'validation_rows': val_count,
        'train_sha256': calc_sha256('train.jsonl'),
        'validation_sha256': calc_sha256('validation.jsonl'),
    },
    'loss_history': {
        'final_train_loss': train_losses[-1] if train_losses else None,
        'final_eval_loss': eval_losses[-1] if eval_losses else None,
        'all_train_losses': train_losses,
        'all_eval_losses': eval_losses,
    },
    'gguf': {
        'filename': target_gguf_filename,
        'sha256': gguf_sha256,
    }
}

with open('train_run.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

!zip -r mockmate_finetuned_artifacts.zip mockmate-llama-adapter mockmate-llama-gguf/mockmate-llama-q4_k_m.gguf train_run.json

print('\nArtifacts zipped successfully!')
print('GGUF SHA256  :', gguf_sha256)
print('\nStarting download...')
files.download('mockmate_finetuned_artifacts.zip')
